In [1]:
import pandas as pd
import numpy as np

In [2]:
df =pd.read_csv("Data set for DADS June.csv")

In [3]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    errors ="coerce",
    dayfirst=True
)
df["Date"].head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         1328 non-null   datetime64[ns]
 1   Time         1328 non-null   object        
 2   Description  1328 non-null   object        
 3   Type         1328 non-null   object        
 4   Amount       1328 non-null   object        
 5   Balance      1328 non-null   float64       
 6   Mode         1328 non-null   object        
 7   Ref          1328 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 83.1+ KB


In [4]:
df["Amount"]=(
    df["Amount"]
    .str.replace("₹","",regex=False)
    .str.replace("Rs.","",regex=False)
    .str.replace(",","",regex=False)
    .str.strip()
)
df["Amount"]= pd.to_numeric(df["Amount"],errors="coerce")


In [5]:
df["Type"]=(
    df["Type"]
    .replace({
        "DR": "debit",
        "Debit":"debit",
        "CR":"credit",
        "Credit":"credit"
    })
)

In [6]:
print("Row before removeing duplicates:",len(df))
print("Duplicate rows:",df.duplicated().sum())
df =df.drop_duplicates()
print("Rows after removing duplicates",len(df))

Row before removeing duplicates: 1328
Duplicate rows: 18
Rows after removing duplicates 1310


In [7]:
print("Number of unique descriptions:",df["Description"].nunique())
df["Description"].unique()[:50]

Number of unique descriptions: 283


array(['AMAZON SELLER SVCS', 'BHIM-BMTC',
       'NEFT-TECHCRUSH LABS-SALARY MAY24', 'UPI-AMAN-8934@OKAXIS',
       'BHIM-BLINKIT', 'BHIM ZEPTO', 'UPI-UBER-2426@HDFCBANK',
       'POS SWIGGY BANGALORE', 'UPI-GROWWPAY@HDFCBANK', 'OLA ELECTRIC',
       'BMS MOVIE TICKETS', 'POS OLA-PRIME', 'SWIGGY-INSTAMART',
       'UPI-STARBUCKS@AXIS', 'UPI-THIRDWAVE@OKAXIS', 'ANI Technologies',
       'BMTC BUS PASS', 'POS TRUFFLES', 'FLIPKART INDIA',
       'POS SWIGGY-RESTAURANT', 'GROFERS INDIA P L', 'POS UBER BANGALORE',
       'BANGALORE ELEC SUPPLY', 'TWC INDIA', 'UPI-BESCOM-BILL@HDFCBANK',
       'UPI-AMAN-0816@OKAXIS', 'ROPPEN TRANSPORTATION', 'OLA CABS',
       'POS ZOMATO', 'UPI-AMAZONPAY@HDFCBANK', 'POS BLINKIT',
       'IMPS-RENT-LANDLORD-75500265', 'ZOMATO MEDIA P L',
       'UPI-ANKIT-6430@OKAXIS', 'UPI-OLACABS@HDFCBANK',
       'UPI-JIORECHARGE@PAYTM', 'UPI-CCD@HDFCBANK', 'Swiggy*Order',
       'INSTAMART BANGALORE', 'UPI-ZOMATO-LIMITED@PAYTM',
       'AVENUE SUPERMARTS', 'POS HP PETROL

In [8]:
vendor_dict = {
    "Swiggy": ["SWIGGY", "BUNDL", "INSTAMART"],
    "Zomato": ["ZOMATO"],
    "Amazon": ["AMAZON", "AMZN"],
    "Blinkit": ["BLINKIT"],
    "Zepto": ["ZEPTO"],
    "Uber": ["UBER"],
    "Ola": ["OLA", "ANI TECHNOLOGIES"],
    "BookMyShow": ["BOOKMYSHOW", "BMS"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRDWAVE", "TWC"],
    "CCD": ["CCD","COFFEE DAY"],
    "Truffles": ["TRUFFLES"],
    "Flipkart": ["FLIPKART"],
    "DMart": ["DMART", "AVENUE SUPERMARTS"],
    "Grofers": ["GROFERS"],
    "HP Petrol": ["HP PETROL"],
    "Jio": ["JIORECHARGE"],
    "Zerodha": ["ZERODHA"],
    "Groww": ["GROWW"],
    "BMTC": ["BMTC"],
    "Bescom": ["BESCOM", "BANGALORE ELEC"],
    "P2P Transfer": [
        "AMAN",
        "ANKIT",
        "PRIYA",
        "VIKAS",
        "RAHUL",
        "NEHA"
    ],
    "Cash Withdrawal": ["ATM-WDL"],
    "Rapido": ["RAPIDO", "ROPPEN"],

    "BigBasket": ["BIGBASKET", "KIRANAKART"],

    "Nykaa": ["NYKAA", "FSN"],

    "Netflix": ["NETFLIX"],

    "Hotstar": ["HOTSTAR", "STAR INDIA"],

    "Airtel": ["AIRTEL"],

    "Indian Oil": ["INDIAN OIL"],

    "BWSSB": ["BWSSB"],

    "Meghana Foods": ["MEGHANA"],

    "Empire Restaurant": ["EMPIRE"],

    "Dineout": ["DINEOUT"],

    "Restaurant": ["BANGALORE RESTAURANT"],

    "Reliance Fresh": ["INNOVATIVE RETAIL"],
    "TechCrush Labs" :["TECHCRUSH"]

}

In [9]:
def extract_vendor(description):
  description = description.upper()
  for vendor, keywords in vendor_dict.items():
    for keyword in keywords:
      if keyword in description:
        return vendor

  return"Uncategorised"



In [10]:
df["vendor_clean"] = df["Description"].apply(extract_vendor)
df[["Description", "vendor_clean"]].head(20)
df["vendor_clean"].value_counts().head(15)

,count
vendor_clean,
Swiggy,243
Zomato,121
Uncategorised,90
Ola,87
Amazon,86
Uber,71
Zepto,58
Rapido,55
Flipkart,43


In [11]:
category_dict = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Blinkit": "Quick Commerce",
    "Zepto": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "BMTC": "Transport",

    "Starbucks": "Cafe",
    "CCD": "Cafe",
    "Third Wave Coffee": "Cafe",

    "Truffles": "Restaurant",

    "BookMyShow": "Entertainment",

    "HP Petrol": "Fuel",

    "Zerodha": "Investment",
    "Groww": "Investment",

    "Bescom": "Utilities",
    "Jio": "Utilities",

    "DMart": "Groceries",
    "Grofers": "Groceries",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal"
}

In [12]:
df["category"]=df["vendor_clean"].map(category_dict)
df[["vendor_clean","category"]].head(28)
df["category"].value_counts()

,count
category,
Food Delivery,364
Transport,195
E-commerce,129
Quick Commerce,98
Cafe,88
Groceries,37
Investment,23
Personal Transfer,17
Cash Withdrawal,17


In [13]:
vendor_spend = df.groupby("vendor_clean")["Amount"].sum()
vendor_spend
vendor_spend.sort_values(ascending=False)

,Amount
vendor_clean,
TechCrush Labs,509774.0
Amazon,328530.0
Uncategorised,246806.0
Zerodha,210000.0
Flipkart,170745.0
Swiggy,104351.0
Zomato,55316.0
Cash Withdrawal,45500.0
Indian Oil,41287.0


In [14]:
df[df["Type"]=="credit"][["Date","Description","Amount"]]

,Date,Description,Amount
2,2024-01-01,NEFT-TECHCRUSH LABS-SALARY MAY24,84728.0
224,2024-02-01,NEFT-TECHCRUSH LABS-SALARY MAY24,84724.0
438,2024-03-01,NEFT-TECHCRUSH LABS-SALARY MAY24,84701.0
658,2024-04-01,NEFT-TECHCRUSH LABS-SALARY MAY24,84736.0
880,2024-05-01,NEFT-TECHCRUSH LABS-SALARY MAY24,85393.0
1113,2024-06-01,NEFT-TECHCRUSH LABS-SALARY MAY24,85492.0


In [15]:
total_debit =df[df["Type"]== "debit"]["Amount"].sum()
total_credit = df[df["Type"] == "credit"]["Amount"].sum()
net_savings = total_credit - total_debit
savings_rate = (net_savings/total_credit)*100

print(f"Total Debit   : ₹{total_debit:,.2f}")
print(f"Total Credit  : ₹{total_credit:,.2f}")
print(f"Net Savings   : ₹{net_savings:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")


Total Debit   : ₹1,678,901.00
Total Credit  : ₹509,774.00
Net Savings   : ₹-1,169,127.00
Savings Rate  : -229.34%


In [16]:
top_vendors =(
    df.groupby("vendor_clean")["Amount"]
    .sum()
    .sort_values(ascending=False)

)
print(top_vendors.head(10))


vendor_clean
TechCrush Labs     509774.0
Amazon             328530.0
Uncategorised      246806.0
Zerodha            210000.0
Flipkart           170745.0
Swiggy             104351.0
Zomato              55316.0
Cash Withdrawal     45500.0
Indian Oil          41287.0
Groww               38160.0
Name: Amount, dtype: float64


In [17]:
df[df["vendor_clean"]=="Uncategorised"]["Description"].value_counts().head(30)

,count
Description,
POS THIRD WAVE COFFEE,11
UPI-MYNTRA@HDFCBANK,10
MYNTRA DESIGNS,5
FKART INTRNET,4
RELIANCE JIO,4
UPI-SPOTIFY-IN@AXIS,3
POS MYNTRA,3
UPI-VI-RECHARGE@HDFCBANK,3
SPOTIFY*PREMIUM,3


In [18]:
print(vendor_dict["Rapido"])
print(vendor_dict["BigBasket"])
print(vendor_dict["TechCrush Labs"])

['RAPIDO', 'ROPPEN']
['BIGBASKET', 'KIRANAKART']
['TECHCRUSH']


In [19]:
print(extract_vendor("UPI-RAPIDO@OKAXIS"))
print(extract_vendor("KIRANAKART TECH"))
print(extract_vendor("NEFT-TECHCRUSH LABS-SALARY MAY24"))
print(extract_vendor("COFFEE DAY GLOBAL"))

Rapido
BigBasket
TechCrush Labs
CCD


In [20]:
vendor_dict.update({

    "Myntra": ["MYNTRA"],

    "Hotstar": ["HOTSTAR", "STAR INDIA"],

    "Netflix": ["NETFLIX"],

    "Indian Oil": ["INDIAN OIL"],

    "Jio": ["JIO"],

    "Airtel": ["AIRTEL"],

    "Restaurant": ["BANGALORE RESTAURANT"],

    "Empire Restaurant": ["EMPIRE"],

    "Meghana Foods": ["MEGHANA"],

    "Dineout": ["DINEOUT"],

    "Nykaa": ["NYKAA", "FSN"],

    "BigBasket": ["BIGBASKET", "KIRANAKART"],

    "Reliance Fresh": ["INNOVATIVE RETAIL"],

    "Flipkart": ["FLIPKART", "FKART"]
})

In [21]:
vendor_dict.update({

    "Third Wave Coffee": ["THIRD WAVE"],

    "Spotify": ["SPOTIFY"],

    "Vi": ["VI POSTPAID", "VODAFONE IDEA", "VI-RECHARGE"],

    "BPCL": ["BPCL"],

    "Indian Oil": ["INDIAN OIL", "IOC"],

    "BookMyShow": ["BIGTREE"],

    "Restaurant": ["RESTAURANT"],

    "Rent": ["RENT-LANDLORD"],

    "P2P Transfer": [
        "KARAN",
        "AMAN",
        "ANKIT",
        "VIKAS",
        "RAHUL",
        "PRIYA",
        "NEHA"
    ]
})

In [22]:
vendor_dict.update({

    "Third Wave Coffee": [
        "THIRDWAVE",
        "THIRD WAVE",
        "TWC"
    ],

    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS",
        "BIGTREE"
    ]

})

In [23]:
category_dict = {

    # Income
    "TechCrush Labs": "Income",

    # Shopping
    "Amazon": "Shopping",
    "Flipkart": "Shopping",
    "Myntra": "Shopping",
    "Nykaa": "Shopping",

    # Food
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",
    "Restaurant": "Dining",
    "Empire Restaurant": "Dining",
    "Meghana Foods": "Dining",
    "CCD": "Cafe",
    "Third Wave Coffee": "Cafe",

    # Groceries
    "Blinkit": "Groceries",
    "BigBasket": "Groceries",
    "DMart": "Groceries",
    "Zepto": "Groceries",
    "Reliance Fresh": "Groceries",

    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    # Fuel
    "Indian Oil": "Fuel",
    "HP Petrol": "Fuel",
    "BPCL": "Fuel",

    # Investments
    "Groww": "Investment",
    "Zerodha": "Investment",

    # Entertainment
    "Netflix": "Entertainment",
    "Hotstar": "Entertainment",
    "BookMyShow": "Entertainment",
    "Spotify": "Entertainment",

    # Utilities
    "Bescom": "Utilities",
    "BWSSB": "Utilities",
    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vi": "Utilities",

    # Others
    "Rent": "Rent",
    "Cash Withdrawal": "Cash Withdrawal",
    "P2P Transfer": "Personal Transfer"

}

In [24]:
category_spend = (
    df.groupby("category")["Amount"]
      .sum()
      .sort_values(ascending=False)
)

print(category_spend)

category
E-commerce           499275.0
Investment           248160.0
Food Delivery        159667.0
Transport             51717.0
Quick Commerce        51261.0
Groceries             46354.0
Cash Withdrawal       45500.0
Utilities             28230.0
Cafe                  27897.0
Personal Transfer     22450.0
Fuel                  17895.0
Restaurant            16673.0
Entertainment          6773.0
Name: Amount, dtype: float64


In [25]:
df["Month"]=df["Date"].dt.month_name()
df[["Date","Month"]].head(10)

,Date,Month
0,2024-01-01,January
1,2024-01-01,January
2,2024-01-01,January
3,2024-01-01,January
4,2024-01-01,January
5,2024-01-01,January
6,2024-01-01,January
7,2024-01-01,January
8,2024-01-02,January
9,2024-01-02,January


In [26]:
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June"
]

In [27]:
df["Month"] = pd.Categorical(
    df["Month"],
    categories=month_order,
    ordered=True
)

In [28]:
monthly_spend = pd.pivot_table(
    df,
    values="Amount",
    index="Month",
    aggfunc="sum"
)

print(monthly_spend)

            Amount
Month             
January   375495.0
February  318795.0
March     415159.0
April     336118.0
May       362699.0
June      380409.0


/tmp/ipykernel_1058/1670491686.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  monthly_spend = pd.pivot_table(


In [32]:
df["Hour"] = df["Time"].str[:2].astype(int)

print(df[["Time", "Hour"]].head())

    Time  Hour
0  03:11     3
1  05:44     5
2  09:35     9
3  14:07    14
4  14:23    14


In [33]:
def get_time_period(hour):

    if 5 <= hour <= 11:
        return "Morning"

    elif 12 <= hour <= 16:
        return "Afternoon"

    elif 17 <= hour <= 20:
        return "Evening"

    else:
        return "Night"

In [34]:
df["Time_Period"] = df["Hour"].apply(get_time_period)
df[["Time", "Hour", "Time_Period"]].head(20)

,Time,Hour,Time_Period
0,03:11,3,Night
1,05:44,5,Morning
2,09:35,9,Morning
3,14:07,14,Afternoon
4,14:23,14,Afternoon
5,14:48,14,Afternoon
6,14:50,14,Afternoon
7,21:44,21,Night
8,05:18,5,Morning
9,06:55,6,Morning


In [35]:
time_spend = (
    df.groupby("Time_Period")["Amount"]
      .sum()
      .sort_values(ascending=False)
)

print(time_spend)

Time_Period
Morning      1010904.0
Afternoon     452169.0
Evening       363884.0
Night         361718.0
Name: Amount, dtype: float64


In [36]:
debit_df = df[df["Type"] == "debit"].copy()

mean = debit_df["Amount"].mean()
std = debit_df["Amount"].std()

debit_df["Z_score"] = (debit_df["Amount"] - mean) / std

anomalies = debit_df[debit_df["Z_score"].abs() > 3]

In [37]:
df["Z_score"] = (df["Amount"] - mean) / std
anomalies = df[df["Z_score"].abs() > 3]
print(anomalies[["Date","Description","Amount","Z_score"]])

           Date                       Description   Amount    Z_score
2    2024-01-01  NEFT-TECHCRUSH LABS-SALARY MAY24  84728.0  29.761943
34   2024-01-05       IMPS-RENT-LANDLORD-75500265  18000.0   5.961092
50   2024-01-07             UPI-ZERODHA-COIN@AXIS  15000.0   4.891038
70   2024-01-09               UPI-MYNTRA@HDFCBANK  10745.0   3.373345
129  2024-01-19              NEFT ZERODHA BROKING  15000.0   4.891038
169  2024-01-24                    FLIPKART INDIA  16260.0   5.340460
171  2024-01-24          POS INDIAN OIL BANGALORE  10408.0   3.253142
224  2024-02-01  NEFT-TECHCRUSH LABS-SALARY MAY24  84724.0  29.760516
248  2024-02-05       IMPS-RENT-LANDLORD-39598076  18000.0   5.961092
269  2024-02-07            UPI-AMAZONPAY@HDFCBANK  21986.0   7.382836
272  2024-02-08                 IMPS ZERODHA-COIN  15000.0   4.891038
336  2024-02-17            UPI-AMAZONPAY@HDFCBANK  13674.0   4.418074
438  2024-03-01  NEFT-TECHCRUSH LABS-SALARY MAY24  84701.0  29.752313
444  2024-03-02     

In [38]:
def spending_archetype(category):

    if category == "Investment":
        return "Investor"

    elif category == "Shopping":
        return "Shopaholic"

    elif category == "Food Delivery":
        return "Foodie"

    elif category == "Transport":
        return "Traveler"

    else:
        return "Balanced Spender"

In [39]:
top_category = category_spend.idxmax()

archetype = spending_archetype(top_category)

print(archetype)

Balanced Spender


In [40]:
recommendations = {

    "Investor":
    "You consistently invest a significant portion of your income. Keep maintaining this disciplined habit while ensuring you have enough emergency savings.",

    "Shopaholic":
    "Shopping is your largest expense. Consider planning purchases and reducing impulse buying to improve your savings.",

    "Foodie":
    "Food delivery is your biggest expense. Cooking at home a few more times each week could reduce your monthly spending.",

    "Traveler":
    "Transportation is your largest expense. Exploring public transport or ride-sharing options may help lower costs.",

    "Balanced Spender":
    "Your spending is well distributed across categories. Continue maintaining this balanced financial behavior."

}

In [41]:
print("="*50)
print("SPENDING ARCHETYPE")
print("="*50)

print("Top Category :", top_category)
print("Archetype    :", archetype)
print()
print("Recommendation:")
print(recommendations[archetype])

SPENDING ARCHETYPE
Top Category : E-commerce
Archetype    : Balanced Spender

Recommendation:
Your spending is well distributed across categories. Continue maintaining this balanced financial behavior.


In [42]:
top_vendor = top_vendors.idxmax()
print(top_vendor)


TechCrush Labs


In [43]:
largest_transaction = df["Amount"].max()
print(largest_transaction)


85492.0


In [44]:
largest_txn = df.loc[df["Amount"].idxmax()]

print(largest_txn)

Date                         2024-06-01 00:00:00
Time                                       09:31
Description     NEFT-TECHCRUSH LABS-SALARY MAY24
Type                                      credit
Amount                                   85492.0
Balance                                 720079.0
Mode                                        NEFT
Ref                                    TXN399607
vendor_clean                      TechCrush Labs
category                                     NaN
Month                                       June
Hour                                           9
Time_Period                              Morning
Z_score                                 30.03445
Name: 1113, dtype: object


In [45]:
num_anomalies = len(anomalies)

print(num_anomalies)

46


In [46]:
print("="*60)
print("BANK TRANSACTION ANALYSIS REPORT")
print("="*60)

print(f"Total Credit        : ₹{total_credit:,.2f}")
print(f"Total Debit         : ₹{total_debit:,.2f}")
print(f"Net Savings         : ₹{net_savings:,.2f}")
print(f"Savings Rate        : {savings_rate:.2f}%")

print("-"*60)

print(f"Top Vendor          : {top_vendor}")
print(f"Top Category        : {top_category}")
print(f"Spending Archetype  : {archetype}")

print("-"*60)

print(f"Largest Transaction : ₹{largest_transaction:,.2f}")
print(f"Number of Anomalies : {num_anomalies}")

print("="*60)

BANK TRANSACTION ANALYSIS REPORT
Total Credit        : ₹509,774.00
Total Debit         : ₹1,678,901.00
Net Savings         : ₹-1,169,127.00
Savings Rate        : -229.34%
------------------------------------------------------------
Top Vendor          : TechCrush Labs
Top Category        : E-commerce
Spending Archetype  : Balanced Spender
------------------------------------------------------------
Largest Transaction : ₹85,492.00
Number of Anomalies : 46
